# Day 1: Graphs and Message Passing Neural Networks

**Geometric Deep Learning Workshop**

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. Represent graphs using adjacency matrices, edge lists, and node features
2. Create and visualize graphs with NetworkX
3. Understand and implement the Message Passing framework
4. Build a Graph Convolutional Network (GCN) layer from scratch in PyTorch
5. Use PyTorch Geometric (PyG) to perform node classification on the Karate Club dataset

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install torch torchvision
# !pip install torch-geometric
# !pip install networkx matplotlib numpy scikit-learn

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import torch
import torch.nn as nn
import torch.nn.functional as F

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

print(f"PyTorch version: {torch.__version__}")
print(f"NetworkX version: {nx.__version__}")

---

## 1. Graph Representations

A graph $G = (V, E)$ consists of a set of **nodes** (vertices) $V$ and **edges** $E$.
There are several standard ways to represent graphs in code.

### 1.1 Adjacency Matrix

For a graph with $n$ nodes, the adjacency matrix $A \in \{0,1\}^{n \times n}$ is defined as:

$$A_{ij} = \begin{cases} 1 & \text{if } (i, j) \in E \\ 0 & \text{otherwise} \end{cases}$$

For undirected graphs, $A$ is symmetric.

In [ ]:
# Create a simple undirected graph with 5 nodes
# Edges: 0-1, 0-2, 1-2, 2-3, 3-4

n_nodes = 5
adj_matrix = np.zeros((n_nodes, n_nodes), dtype=np.float32)

# Add edges (undirected, so symmetric)
edges = [(0, 1), (0, 2), (1, 2), (2, 3), (3, 4)]
for i, j in edges:
    adj_matrix[i, j] = 1.0
    adj_matrix[j, i] = 1.0

print("Adjacency Matrix:")
print(adj_matrix)
print(f"\nShape: {adj_matrix.shape}")
print(f"Symmetric: {np.allclose(adj_matrix, adj_matrix.T)}")

### 1.2 Edge List

An edge list stores only the existing edges, which is more memory-efficient for sparse graphs.
In PyTorch Geometric, this is stored as a `[2, num_edges]` tensor (COO format).

In [ ]:
# Edge list representation
edge_list = np.array(edges)
print("Edge list (each row is an edge):")
print(edge_list)

# COO format used by PyTorch Geometric: [2, num_edges]
# For undirected graphs, we include both directions
src = [i for i, j in edges] + [j for i, j in edges]
dst = [j for i, j in edges] + [i for i, j in edges]
edge_index = np.array([src, dst])

print(f"\nCOO edge_index (shape {edge_index.shape}):")
print(edge_index)

# Verify: we can reconstruct the adjacency matrix from the edge index
adj_reconstructed = np.zeros((n_nodes, n_nodes))
for k in range(edge_index.shape[1]):
    adj_reconstructed[edge_index[0, k], edge_index[1, k]] = 1.0
print(f"\nReconstruction matches: {np.allclose(adj_matrix, adj_reconstructed)}")

### 1.3 Node Features

In many applications, each node has an associated feature vector $x_i \in \mathbb{R}^d$.
The node feature matrix is $X \in \mathbb{R}^{n \times d}$.

In [ ]:
# Create random node features (5 nodes, 3 features each)
feature_dim = 3
node_features = np.random.randn(n_nodes, feature_dim).astype(np.float32)

print("Node Feature Matrix X:")
print(node_features)
print(f"Shape: {node_features.shape} (n_nodes x feature_dim)")

# The degree matrix D is diagonal with D_ii = sum of row i in A
degree = adj_matrix.sum(axis=1)
D = np.diag(degree)
print(f"\nDegree of each node: {degree}")
print(f"Degree Matrix D:\n{D}")

---

## 2. NetworkX Basics

NetworkX is a Python library for creating, manipulating, and studying complex networks.

In [ ]:
# Create the same graph in NetworkX
G = nx.Graph()
G.add_nodes_from(range(5))
G.add_edges_from(edges)

print(f"Number of nodes: {G.number_of_nodes()}")
print(f"Number of edges: {G.number_of_edges()}")
print(f"Nodes: {list(G.nodes())}")
print(f"Edges: {list(G.edges())}")

In [ ]:
# Visualize the graph
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Layout 1: Spring layout (force-directed)
pos_spring = nx.spring_layout(G, seed=42)
nx.draw(G, pos_spring, ax=axes[0], with_labels=True, node_color='steelblue',
        node_size=500, font_color='white', font_weight='bold', edge_color='gray')
axes[0].set_title('Spring Layout')

# Layout 2: Circular layout
pos_circ = nx.circular_layout(G)
nx.draw(G, pos_circ, ax=axes[1], with_labels=True, node_color='coral',
        node_size=500, font_color='white', font_weight='bold', edge_color='gray')
axes[1].set_title('Circular Layout')

# Layout 3: Kamada-Kawai layout
pos_kk = nx.kamada_kawai_layout(G)
nx.draw(G, pos_kk, ax=axes[2], with_labels=True, node_color='seagreen',
        node_size=500, font_color='white', font_weight='bold', edge_color='gray')
axes[2].set_title('Kamada-Kawai Layout')

plt.tight_layout()
plt.show()

In [ ]:
# Basic graph metrics
print("=== Graph Metrics ===")
print(f"Density: {nx.density(G):.4f}")
print(f"Is connected: {nx.is_connected(G)}")
print(f"Diameter: {nx.diameter(G)}")
print(f"Average shortest path length: {nx.average_shortest_path_length(G):.4f}")
print(f"Average clustering coefficient: {nx.average_clustering(G):.4f}")

print("\n=== Node-level Metrics ===")
print(f"Degree centrality: {nx.degree_centrality(G)}")
print(f"Betweenness centrality: {nx.betweenness_centrality(G)}")
print(f"Closeness centrality: {nx.closeness_centrality(G)}")

# Degree distribution
degrees = [d for _, d in G.degree()]
print(f"\nDegree sequence: {degrees}")

In [ ]:
# Visualize a more interesting graph: Karate Club
G_karate = nx.karate_club_graph()

# Color nodes by their club membership
colors = ['steelblue' if G_karate.nodes[n]['club'] == 'Mr. Hi'
          else 'coral' for n in G_karate.nodes()]

fig, ax = plt.subplots(figsize=(10, 7))
pos = nx.spring_layout(G_karate, seed=42)
nx.draw(G_karate, pos, ax=ax, with_labels=True, node_color=colors,
        node_size=400, font_size=9, font_color='white',
        font_weight='bold', edge_color='lightgray', width=0.8)
ax.set_title("Zachary's Karate Club (colored by ground-truth community)", fontsize=14)
plt.tight_layout()
plt.show()

print(f"Nodes: {G_karate.number_of_nodes()}, Edges: {G_karate.number_of_edges()}")

---

## 3. The Message Passing Framework

Most GNNs follow the **Message Passing Neural Network (MPNN)** framework (Gilmer et al., 2017).

At each layer $\ell$, the representation of node $i$ is updated as:

$$h_i^{(\ell+1)} = \text{UPDATE}\left(h_i^{(\ell)},\; \text{AGGREGATE}\left(\{\text{MESSAGE}(h_i^{(\ell)}, h_j^{(\ell)}, e_{ij}) : j \in \mathcal{N}(i)\}\right)\right)$$

Three key operations:
1. **MESSAGE**: Compute a message from neighbor $j$ to node $i$
2. **AGGREGATE**: Combine all incoming messages (must be permutation-invariant: sum, mean, max)
3. **UPDATE**: Update the node representation using the aggregated message

In [ ]:
# Illustrate message passing step-by-step with numpy

# Our graph from above:
# Nodes: 0, 1, 2, 3, 4
# Edges: 0-1, 0-2, 1-2, 2-3, 3-4

print("Initial node features:")
for i in range(n_nodes):
    print(f"  Node {i}: {node_features[i]}")

# Step 1: MESSAGE - identity function (just pass neighbor features)
print("\n--- Message Passing Round ---")
print("\nStep 1 - Messages (neighbor features):")
for i in range(n_nodes):
    neighbors = np.where(adj_matrix[i] == 1)[0]
    print(f"  Node {i} receives from neighbors {list(neighbors)}")

# Step 2: AGGREGATE - mean of neighbor features
print("\nStep 2 - Aggregation (mean of neighbor features):")
aggregated = np.zeros_like(node_features)
for i in range(n_nodes):
    neighbors = np.where(adj_matrix[i] == 1)[0]
    if len(neighbors) > 0:
        aggregated[i] = node_features[neighbors].mean(axis=0)
    print(f"  Node {i}: aggregated = {aggregated[i]}")

# Step 3: UPDATE - concatenate original + aggregated, then linear transform
print("\nStep 3 - Update (concat + linear transform):")
combined = np.concatenate([node_features, aggregated], axis=1)  # [n, 2d]
W_update = np.random.randn(2 * feature_dim, feature_dim).astype(np.float32) * 0.5
updated_features = np.maximum(0, combined @ W_update)  # ReLU activation

for i in range(n_nodes):
    print(f"  Node {i}: {updated_features[i]}")

print(f"\nOutput shape: {updated_features.shape}")

---

## 4. GCN from Scratch in PyTorch

The **Graph Convolutional Network** (Kipf & Welling, 2017) uses the following layer:

$$H^{(\ell+1)} = \sigma\left(\tilde{D}^{-1/2} \tilde{A} \tilde{D}^{-1/2} H^{(\ell)} W^{(\ell)}\right)$$

where:
- $\tilde{A} = A + I$ (adjacency with self-loops)
- $\tilde{D}_{ii} = \sum_j \tilde{A}_{ij}$ (degree matrix of $\tilde{A}$)
- $W^{(\ell)}$ is a learnable weight matrix
- $\sigma$ is an activation function (e.g., ReLU)

In [ ]:
class GCNLayer(nn.Module):
    """A single Graph Convolutional Network layer (Kipf & Welling, 2017)."""

    def __init__(self, in_features: int, out_features: int, bias: bool = True):
        super().__init__()
        self.weight = nn.Parameter(torch.FloatTensor(in_features, out_features))
        if bias:
            self.bias = nn.Parameter(torch.FloatTensor(out_features))
        else:
            self.bias = None
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.xavier_uniform_(self.weight)
        if self.bias is not None:
            nn.init.zeros_(self.bias)

    def forward(self, x: torch.Tensor, adj: torch.Tensor) -> torch.Tensor:
        """
        Forward pass.

        Args:
            x: Node feature matrix [n_nodes, in_features]
            adj: Adjacency matrix [n_nodes, n_nodes]

        Returns:
            Updated node features [n_nodes, out_features]
        """
        # Step 1: Add self-loops -> A_tilde = A + I
        n = adj.size(0)
        adj_hat = adj + torch.eye(n, device=adj.device)

        # Step 2: Compute degree matrix D_tilde and its inverse square root
        degree = adj_hat.sum(dim=1)  # [n]
        d_inv_sqrt = torch.diag(degree.pow(-0.5))  # D^{-1/2}

        # Step 3: Symmetric normalization: D^{-1/2} A_hat D^{-1/2}
        adj_norm = d_inv_sqrt @ adj_hat @ d_inv_sqrt

        # Step 4: Linear transform then propagate: A_norm @ X @ W
        support = x @ self.weight  # [n, out_features]
        output = adj_norm @ support  # [n, out_features]

        if self.bias is not None:
            output = output + self.bias

        return output


# Test the layer
X = torch.from_numpy(node_features)  # [5, 3]
A = torch.from_numpy(adj_matrix)      # [5, 5]

gcn_layer = GCNLayer(in_features=3, out_features=4)
output = gcn_layer(X, A)

print(f"Input shape:  {X.shape}")
print(f"Output shape: {output.shape}")
print(f"Output:\n{output.detach().numpy()}")

In [ ]:
class GCN(nn.Module):
    """Two-layer GCN for node classification."""

    def __init__(self, in_features: int, hidden_dim: int, num_classes: int,
                 dropout: float = 0.5):
        super().__init__()
        self.gc1 = GCNLayer(in_features, hidden_dim)
        self.gc2 = GCNLayer(hidden_dim, num_classes)
        self.dropout = dropout

    def forward(self, x: torch.Tensor, adj: torch.Tensor) -> torch.Tensor:
        # Layer 1: GCN + ReLU + Dropout
        x = self.gc1(x, adj)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        # Layer 2: GCN (no activation, raw logits for classification)
        x = self.gc2(x, adj)
        return x


# Build a GCN for our toy graph: 3 input features -> 8 hidden -> 2 classes
model = GCN(in_features=3, hidden_dim=8, num_classes=2)
print(model)

# Forward pass
logits = model(X, A)
print(f"\nLogits shape: {logits.shape}")
print(f"Predictions: {logits.argmax(dim=1).tolist()}")

---

## 5. Introduction to PyTorch Geometric (PyG)

PyTorch Geometric provides optimized implementations of GNNs using sparse operations.
The core data structure is `torch_geometric.data.Data`.

In [ ]:
import torch_geometric
from torch_geometric.data import Data

print(f"PyTorch Geometric version: {torch_geometric.__version__}")

# Create a PyG Data object for our toy graph
edge_index_tensor = torch.tensor([
    [0, 0, 1, 1, 2, 2, 2, 3, 3, 4],  # source nodes
    [1, 2, 0, 2, 0, 1, 3, 2, 4, 3],  # target nodes
], dtype=torch.long)

x = torch.randn(5, 3)  # 5 nodes, 3 features each
y = torch.tensor([0, 0, 1, 1, 1])  # node labels (2 classes)

data = Data(x=x, edge_index=edge_index_tensor, y=y)

print("\nPyG Data object:")
print(f"  {data}")
print(f"  Number of nodes: {data.num_nodes}")
print(f"  Number of edges: {data.num_edges}")
print(f"  Number of node features: {data.num_node_features}")
print(f"  Has isolated nodes: {data.has_isolated_nodes()}")
print(f"  Has self-loops: {data.has_self_loops()}")
print(f"  Is undirected: {data.is_undirected()}")

In [ ]:
# Converting between NetworkX and PyG
from torch_geometric.utils import from_networkx, to_networkx

# NetworkX -> PyG
G_small = nx.Graph()
G_small.add_edges_from([(0, 1), (1, 2), (2, 3), (3, 0)])

# Assign node features as NetworkX node attributes
for node in G_small.nodes():
    G_small.nodes[node]['x'] = [float(node), float(node ** 2)]

pyg_data = from_networkx(G_small, group_node_attrs=['x'])
print("Converted from NetworkX:")
print(f"  {pyg_data}")
print(f"  Node features:\n{pyg_data.x}")

# PyG -> NetworkX
G_back = to_networkx(pyg_data, to_undirected=True)
print(f"\nConverted back to NetworkX: {G_back.number_of_nodes()} nodes, {G_back.number_of_edges()} edges")

---

## 6. Node Classification on Karate Club with PyG

Let us put everything together. We will use PyG's built-in `GCNConv` layer
to classify nodes in Zachary's Karate Club into the two factions.

In [ ]:
from torch_geometric.datasets import KarateClub
from torch_geometric.nn import GCNConv
from torch_geometric.utils import to_networkx

# Load the Karate Club dataset
dataset = KarateClub()
data = dataset[0]

print(f"Dataset: {dataset}")
print(f"Number of graphs: {len(dataset)}")
print(f"Number of features: {dataset.num_features}")
print(f"Number of classes: {dataset.num_classes}")
print(f"\nGraph:")
print(f"  Nodes: {data.num_nodes}")
print(f"  Edges: {data.num_edges}")
print(f"  Train mask: {data.train_mask.sum().item()} labeled nodes")
print(f"  Labels: {data.y.tolist()}")

In [ ]:
class GCN_PyG(nn.Module):
    """GCN for node classification using PyG's GCNConv."""

    def __init__(self, in_channels: int, hidden_channels: int,
                 out_channels: int):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        # First GCN layer
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        # Second GCN layer
        x = self.conv2(x, edge_index)
        return x


model_pyg = GCN_PyG(
    in_channels=dataset.num_features,
    hidden_channels=16,
    out_channels=dataset.num_classes,
)
print(model_pyg)

In [ ]:
# Training loop
optimizer = torch.optim.Adam(model_pyg.parameters(), lr=0.01, weight_decay=5e-4)
criterion = nn.CrossEntropyLoss()

losses = []
accuracies = []

model_pyg.train()
for epoch in range(200):
    optimizer.zero_grad()
    out = model_pyg(data.x, data.edge_index)  # Forward pass

    # Loss only on labeled (training) nodes
    loss = criterion(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()

    # Compute accuracy on ALL nodes
    pred = out.argmax(dim=1)
    acc = (pred == data.y).float().mean().item()

    losses.append(loss.item())
    accuracies.append(acc)

    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1:3d} | Loss: {loss.item():.4f} | Accuracy: {acc:.4f}")

print(f"\nFinal accuracy: {accuracies[-1]:.4f}")

In [ ]:
# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(losses, color='steelblue')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss')
ax1.grid(True, alpha=0.3)

ax2.plot(accuracies, color='coral')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Node Classification Accuracy (all nodes)')
ax2.set_ylim(0, 1.05)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Visualize the learned node embeddings using the hidden layer
model_pyg.eval()

# Get embeddings from the first GCN layer
with torch.no_grad():
    h = model_pyg.conv1(data.x, data.edge_index)
    h = F.relu(h)

# Use t-SNE or PCA to reduce to 2D for visualization
from sklearn.manifold import TSNE

embeddings = h.numpy()
tsne = TSNE(n_components=2, random_state=42, perplexity=10)
emb_2d = tsne.fit_transform(embeddings)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Colored by ground truth labels
scatter1 = ax1.scatter(emb_2d[:, 0], emb_2d[:, 1], c=data.y.numpy(),
                       cmap='Set1', s=80, edgecolors='black', linewidths=0.5)
for i in range(data.num_nodes):
    ax1.annotate(str(i), (emb_2d[i, 0], emb_2d[i, 1]), fontsize=7,
                 ha='center', va='center')
ax1.set_title('GCN Embeddings (colored by ground truth)')
ax1.set_xlabel('t-SNE dim 1')
ax1.set_ylabel('t-SNE dim 2')

# Plot 2: Colored by predictions
with torch.no_grad():
    preds = model_pyg(data.x, data.edge_index).argmax(dim=1).numpy()

scatter2 = ax2.scatter(emb_2d[:, 0], emb_2d[:, 1], c=preds,
                       cmap='Set1', s=80, edgecolors='black', linewidths=0.5)
for i in range(data.num_nodes):
    ax2.annotate(str(i), (emb_2d[i, 0], emb_2d[i, 1]), fontsize=7,
                 ha='center', va='center')
ax2.set_title('GCN Embeddings (colored by predictions)')
ax2.set_xlabel('t-SNE dim 1')
ax2.set_ylabel('t-SNE dim 2')

plt.tight_layout()
plt.show()

---

## Summary

In this notebook we covered:

| Topic | Key Takeaway |
|-------|-------------|
| Graph representations | Adjacency matrix, edge list (COO), node features |
| NetworkX | Graph creation, visualization, and metrics |
| Message Passing | MESSAGE, AGGREGATE, UPDATE framework |
| GCN from scratch | Symmetric normalization: $\tilde{D}^{-1/2}\tilde{A}\tilde{D}^{-1/2}XW$ |
| PyG Data objects | `Data(x=..., edge_index=..., y=...)` |
| Node classification | Semi-supervised learning on Karate Club |

**Next**: Day 2 will cover spectral methods, attention mechanisms (GAT), and graph classification.

---

### Exercises

1. **Modify the GCN**: Add a third layer to the scratch GCN. Does it improve accuracy? Why or why not?
2. **Different aggregation**: Modify the message passing example to use `max` instead of `mean` aggregation. How do the results differ?
3. **Edge features**: Create a PyG Data object with edge attributes. Hint: use `data.edge_attr`.
4. **Experiment**: Try different hidden dimensions and learning rates on Karate Club. What works best?